# FinIA-Flex — Paso 7: Filtro de Calidad Automático

Caso Práctico Unidad 1, materia Generative IA, IEP.

Este paso cumple el requisito 7 del caso: filtrar y controlar la calidad de lo que genera el
modelo. No lo armé con una lista genérica de buenas prácticas — lo construí directo a partir
de los hallazgos reales que ya tuve en los Pasos 4 y 5:

| # | Lo que encontré | Qué reviso aquí |
|---|---|---|
| 1 | El modelo inventó un responsable que no le di (Paso 4) | Que el responsable citado sea exactamente el que puse en los datos |
| 2 | Comparó el gasto real contra el umbral en vez de la variación (Paso 5) | Que lo que diga sobre "excede el umbral" coincida con lo que ya calculé en código |
| 3 | (por si acaso) que cite una política que no está en el contexto | Que cualquier código de política citado sí venga del contexto recuperado |

También reviso que tenga las 5 secciones obligatorias.

Para probar que de verdad funciona, en la Sección 4 lo corro contra los textos defectuosos
reales que ya generé antes — no ejemplos inventados.


## 1. Configuración del entorno

Se reutilizan los mismos componentes construidos en el Paso 5 (índice vectorial, prompt maestro, cálculo de indicadores).

In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-text-splitters chromadb sentence-transformers groq

from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Ingresar API key de Groq: ")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_PATH = "/content/drive/MyDrive/FinIA-Flex/finia_flex_chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)
print(f"Índice cargado con {vectorstore._collection.count()} fragmentos disponibles.")

### Funciones del Paso 5 (cálculo de indicadores, recuperación y generación)

Sin cambios respecto a la versión ya validada.

In [ ]:
UMBRALES_APROBACION_MXN = {
    "Materia Prima": 80000, "Mano de Obra Directa": 40000, "Energía": 30000,
    "Mantenimiento": 50000, "Logística/Fletes": 35000,
}

def clasificar_variacion(variacion_pct: float) -> str:
    if variacion_pct >= 10: return "Variación significativa"
    elif variacion_pct >= 3: return "Variación moderada"
    elif variacion_pct >= -2.9: return "Dentro de rango normal"
    elif variacion_pct >= -10: return "Ahorro saludable"
    else: return "Ahorro atípico"

def calcular_racha_sobrecosto(variacion_pct_actual, historico_mensual):
    racha = 1 if variacion_pct_actual > 0 else 0
    if racha == 0: return 0
    for _, pct in reversed(historico_mensual):
        if pct > 0: racha += 1
        else: break
    return racha

def calcular_indicadores(categoria, variacion_mxn, variacion_pct, historico_mensual=None):
    historico_mensual = historico_mensual or []
    umbral = UMBRALES_APROBACION_MXN.get(categoria)
    excede_umbral = (umbral is not None) and (variacion_mxn > umbral)
    racha = calcular_racha_sobrecosto(variacion_pct, historico_mensual)
    return {
        "clasificacion": clasificar_variacion(variacion_pct),
        "umbral_categoria_mxn": umbral,
        "excede_umbral": excede_umbral,
        "racha_meses_sobrecosto": racha,
        "variacion_sostenida": racha >= 3,
    }

def recuperar_contexto(centro_costo, categoria, resumen_situacion, historico="", k=3):
    consulta = (
        f"Política de aprobación y control de gasto para la categoría {categoria} "
        f"en el centro de costo {centro_costo}. Situación actual: {resumen_situacion}. "
        f"Histórico reciente: {historico}. "
        f"¿Qué umbral de aprobación o regla de variación sostenida aplica?"
    )
    resultados = vectorstore.similarity_search(consulta, k=k)
    fragmentos_texto = []
    for r in resultados:
        fuente = r.metadata["source"].split("/")[-1]
        fragmentos_texto.append(f'({fuente})\n"{r.page_content}"')
    return "\n\n".join(fragmentos_texto)

print("Funciones del Paso 5 cargadas.")

## 2. La función de validación

Esta función recibe el texto que generó el LLM más los datos e indicadores reales, y revisa 5
puntos sin volver a llamar al modelo — puras comparaciones de texto y números. Esto es a
propósito: no le pido al LLM que se revise a sí mismo, porque ya vi que no es confiable para
eso.


In [ ]:
import re

def _oraciones_con_excede(texto: str) -> list:
    """Encuentra oraciones donde aparece una forma NO negada del verbo 'exceder'."""
    resultados = []
    for m in re.finditer(r"exced\w*", texto, re.IGNORECASE):
        inicio = m.start()
        antes = texto[max(0, inicio - 15):inicio].lower()
        if "no " in antes or "sin " in antes:
            continue  # negacion detectada justo antes -> no cuenta como afirmacion positiva
        ini_oracion = texto.rfind(".", 0, inicio) + 1
        fin_oracion = texto.find(".", inicio)
        fin_oracion = fin_oracion if fin_oracion != -1 else len(texto)
        resultados.append(texto[ini_oracion:fin_oracion + 1])
    return resultados


def afirma_exceso_umbral(texto: str, umbral_mxn) -> bool:
    """
    Determina si el texto afirma (sin negar) que se excede el umbral, aceptando
    cualquier conjugacion del verbo (excede, excedio, excediendo, exceden)
    y reconociendo tanto la mencion explicita de "umbral" como el monto exacto.
    """
    for oracion in _oraciones_con_excede(texto):
        oracion_low = oracion.lower()
        if "umbral" in oracion_low:
            return True
        montos = [float(x.replace(",", "")) for x in re.findall(r"\$([\d,]+(?:\.\d+)?)", oracion)]
        if umbral_mxn and any(abs(m - umbral_mxn) < 1 for m in montos):
            return True
    return False


def validar_reporte(texto_reporte: str, responsable_esperado: str, variacion_mxn: float,
                     indicadores: dict, contexto_recuperado: str) -> dict:
    """
    Audita el reporte generado por el LLM contra los datos de entrada, sin usar el LLM
    para la verificación. Devuelve un diccionario con el resultado de cada chequeo.
    """
    problemas = []

    # 1. Estructura obligatoria de 5 secciones
    secciones_esperadas = [
        "1. Resumen Ejecutivo", "2. Diagnóstico por Centro de Costo",
        "3. Alertas de Política", "4. Recomendación", "5. Responsable y Siguiente Paso",
    ]
    secciones_faltantes = [s for s in secciones_esperadas if s not in texto_reporte]
    if secciones_faltantes:
        problemas.append(f"Estructura incompleta. Faltan secciones: {secciones_faltantes}")

    # 2. Responsable: debe coincidir exactamente con el dato de entrada
    match_responsable = re.search(r"Responsable:\s*(.+?)(?:\n|$)", texto_reporte)
    responsable_citado = match_responsable.group(1).strip() if match_responsable else None
    if responsable_citado is None:
        problemas.append("No se encontró el campo Responsable en el texto generado.")
    elif responsable_esperado.lower() not in responsable_citado.lower() and \
         responsable_citado.lower() not in responsable_esperado.lower():
        problemas.append(
            f'Responsable inconsistente: el reporte dice "{responsable_citado}", '
            f'pero el dato de entrada era "{responsable_esperado}".'
        )

    # 3. Consistencia numérica: la variación en MXN calculada debe aparecer en el texto
    variacion_abs = abs(variacion_mxn)
    montos_en_texto = [
        float(m.replace(",", "")) for m in re.findall(r"\$([\d,]+(?:\.\d+)?)", texto_reporte)
    ]
    variacion_presente = any(abs(m - variacion_abs) < 1 for m in montos_en_texto)
    if not variacion_presente:
        problemas.append(
            f"La variación calculada (${variacion_abs:,.0f} MXN) no aparece explícitamente "
            f"en el texto generado — posible inconsistencia numérica."
        )

    # 4. Consistencia de alertas: lo que el texto afirma debe coincidir con los indicadores
    afirma_excede_umbral = afirma_exceso_umbral(texto_reporte, indicadores["umbral_categoria_mxn"])
    if afirma_excede_umbral and not indicadores["excede_umbral"]:
        problemas.append(
            "El texto afirma que se excede el umbral de aprobación, pero el indicador "
            "calculado en código dice que NO se excede — inconsistencia numérica grave."
        )
    if indicadores["excede_umbral"] and not afirma_excede_umbral:
        problemas.append(
            "El indicador calculado señala que SÍ se excede el umbral de aprobación, pero "
            "el texto generado no lo menciona — alerta de política potencialmente omitida."
        )

    afirma_sostenida = bool(re.search(r"variaci[oó]n\s+sostenida", texto_reporte, re.IGNORECASE))
    if indicadores["variacion_sostenida"] and not afirma_sostenida:
        problemas.append(
            "El indicador calculado señala que SÍ aplica la regla de variación sostenida, "
            "pero el texto generado no la menciona — alerta de política potencialmente omitida."
        )

    # 5. Citas de política: cualquier código POL-FIN-XXX citado debe existir en el contexto recuperado
    codigos_citados = set(re.findall(r"POL-FIN-\d{3}", texto_reporte))
    codigos_en_contexto = set(re.findall(r"POL-FIN-\d{3}", contexto_recuperado))
    codigos_no_respaldados = codigos_citados - codigos_en_contexto
    if codigos_no_respaldados:
        problemas.append(
            f"Se citan códigos de política no presentes en el contexto recuperado: "
            f"{codigos_no_respaldados} — posible alucinación de fuente."
        )

    return {
        "aprobado": len(problemas) == 0,
        "problemas": problemas,
    }

print("Función validar_reporte() lista.")

## 3. Uniendo todo: generar + validar

Envuelvo `generar_reporte()` del Paso 5 con esta validación. Si el reporte no pasa algún
chequeo, muestra la advertencia — en un sistema real, eso se marcaría para revisión humana en
vez de mostrarse tal cual.


In [ ]:
from groq import Groq

client = Groq()

SYSTEM_PROMPT = """Eres un analista financiero senior de FlexParts Manufacturing MX,
especializado en control de costos de manufactura. Tu tarea es generar reportes ejecutivos
de variación presupuestal para Gerencia, a partir de datos de presupuesto vs. gasto real y
del contexto de políticas internas que se te proporcione.

RAZONAMIENTO INTERNO (no lo muestres en la respuesta final, solo úsalo para pensar):
1. Los indicadores de clasificación, umbral y variación sostenida ya vienen calculados por
   el sistema en el bloque "INDICADORES CALCULADOS POR EL SISTEMA". NO los recalcules, NO los
   contradigas, NO compares tú mismo montos contra umbrales — usa esos valores tal cual.
2. Con base en esos indicadores ya calculados, verifica si el contexto de políticas
   proporcionado incluye una regla que corresponda a lo que los indicadores señalan (umbral
   excedido, variación sostenida, o ninguno). Si el contexto no incluye una política aplicable
   a lo que indican los indicadores, dilo explícitamente — nunca inventes un umbral o una
   regla que no esté en el contexto.
3. Distingue causas internas (atendibles por el responsable del centro de costo) de causas
   externas (fuera de su control), cuando el contexto lo permita.
4. Solo después de este análisis, redacta el reporte final.

ESTRUCTURA OBLIGATORIA DE LA RESPUESTA FINAL:
1. Resumen Ejecutivo (máximo 3 líneas)
2. Diagnóstico por Centro de Costo (variación en MXN y %, clasificación, causa probable)
3. Alertas de Política (solo si el contexto proporcionado activa alguna; si no, escribir
   "Sin alertas de política en el contexto disponible")
4. Recomendación (una acción concreta y accionable por cada hallazgo relevante)
5. Responsable y Siguiente Paso

REGLAS DE GROUNDING:
- Usa exclusivamente los datos numéricos y el contexto de políticas que se te proporcionen.
- Si citas una política o un umbral, debe provenir textualmente del contexto recibido.
- Si el contexto no cubre algo que sería útil mencionar, indica la limitación en vez de
  completar con supuestos.
- NUNCA inventes nombres de personas, cargos o responsables. El campo "Responsable" debe
  llenarse únicamente con el valor recibido en los DATOS de entrada, copiado tal cual. Si el
  campo Responsable no viene incluido en los DATOS, escribe exactamente "No especificado en
  los datos proporcionados" — no propongas un nombre, cargo o departamento por tu cuenta bajo
  ninguna circunstancia.
- NUNCA recalcules ni contradigas los valores del bloque "INDICADORES CALCULADOS POR EL
  SISTEMA". Si ese bloque indica que el umbral NO se excedió, no afirmes lo contrario aunque
  el monto te parezca alto; si indica que sí se excedió, no lo minimices.

TONO: profesional, directo, sin tecnicismos innecesarios. El reporte debe ser legible para
un Gerente de Planta que no es especialista financiero. Evita juicios de valor sobre las
personas; evalúa procesos y resultados.
"""

EJEMPLO_FEW_SHOT_ENTRADA = """
DATOS:
Centro de costo: Línea de Producción 2
Categoría: Mantenimiento
Presupuesto: $70,000 MXN | Real: $87,200 MXN (Septiembre)
Histórico: Julio +18.6%, Agosto +20.0%, Septiembre +24.6%
Responsable: Coordinador de Mantenimiento - J. Salinas

CONTEXTO DE POLÍTICAS RECUPERADO:
"Cuando una categoría de gasto en un mismo centro de costo presenta una variación positiva
(sobrecosto) durante 3 meses consecutivos o más, el responsable debe presentar un plan
correctivo formal a Gerencia, independientemente de si cada mes individual superó o no el
umbral de aprobación." (Política POL-FIN-001, sección 4)
"""

EJEMPLO_FEW_SHOT_SALIDA = """
1. Resumen Ejecutivo
Mantenimiento en Línea de Producción 2 muestra sobrecosto sostenido por tercer mes
consecutivo, activando la regla de variación sostenida de la Política POL-FIN-001.

2. Diagnóstico por Centro de Costo
- Línea de Producción 2 / Mantenimiento: variación de +$17,200 MXN (+24.6%) en septiembre.
  Clasificación: significativa. Tendencia sostenida desde julio (+18.6%, +20.0%, +24.6%).

3. Alertas de Política
Se activa la regla de variación sostenida (POL-FIN-001, sección 4): 3 meses consecutivos de
sobrecosto en la misma categoría y centro de costo requieren plan correctivo formal a
Gerencia, independientemente del monto individual de cada mes.

4. Recomendación
Solicitar al Coordinador de Mantenimiento un plan correctivo formal antes del cierre del
siguiente mes, desagregando el gasto entre mantenimiento correctivo y preventivo para
identificar si el sobrecosto responde a fallas puntuales o a un patrón estructural.

5. Responsable y Siguiente Paso
Responsable: Coordinador de Mantenimiento - J. Salinas.
Siguiente paso: presentar plan correctivo formal a Gerencia — fecha límite sugerida: cierre
del mes en curso.
"""


def generar_reporte_validado(centro_costo, categoria, presupuesto, real, historico,
                              historico_mensual=None, responsable=None):
    variacion_mxn = real - presupuesto
    variacion_pct = variacion_mxn / presupuesto * 100
    indicadores = calcular_indicadores(categoria, variacion_mxn, variacion_pct, historico_mensual)

    resumen_situacion = f"variación de {variacion_pct:.1f}% ({variacion_mxn:,.0f} MXN)"
    contexto = recuperar_contexto(centro_costo, categoria, resumen_situacion, historico=historico)

    responsable_texto = responsable if responsable else "No especificado en los datos proporcionados"
    umbral_texto = (f"${indicadores['umbral_categoria_mxn']:,.0f} MXN"
                     if indicadores["umbral_categoria_mxn"] else "sin umbral definido para esta categoría")

    caso_entrada = f"""
DATOS:
Centro de costo: {centro_costo}
Categoría: {categoria}
Presupuesto: ${presupuesto:,.0f} MXN | Real: ${real:,.0f} MXN
Variación: {variacion_pct:.1f}% (${variacion_mxn:,.0f} MXN)
Histórico: {historico}
Responsable: {responsable_texto}

INDICADORES CALCULADOS POR EL SISTEMA (usa estos valores tal cual, no los recalcules ni los
contradigas — fueron calculados por código, no por ti):
- Clasificación de la variación: {indicadores['clasificacion']}
- Umbral de aprobación de la categoría "{categoria}": {umbral_texto}
- ¿La variación de este mes excede el umbral de su categoría?: {"SÍ" if indicadores['excede_umbral'] else "NO"}
- Meses consecutivos de sobrecosto (incluyendo el actual): {indicadores['racha_meses_sobrecosto']}
- ¿Aplica la regla de variación sostenida (3+ meses consecutivos de sobrecosto)?: {"SÍ" if indicadores['variacion_sostenida'] else "NO"}

CONTEXTO DE POLÍTICAS RECUPERADO (automático):
{contexto}
"""

    respuesta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": EJEMPLO_FEW_SHOT_ENTRADA},
            {"role": "assistant", "content": EJEMPLO_FEW_SHOT_SALIDA},
            {"role": "user", "content": caso_entrada},
        ],
        temperature=0.3,
    )
    texto_reporte = respuesta.choices[0].message.content

    resultado_validacion = validar_reporte(
        texto_reporte, responsable_texto, variacion_mxn, indicadores, contexto
    )

    return texto_reporte, resultado_validacion


print("Función generar_reporte_validado() lista.")

## 4. Probando el filtro contra mis propios errores

Antes de probar con casos nuevos, corro `validar_reporte()` contra los textos defectuosos
reales que ya me generó el sistema en los Pasos 4 y 5 — para confirmar que sí los hubiera
atrapado si hubiera existido en ese momento.


In [ ]:
# Hallazgo 1 (Paso 4): responsable inventado
texto_defectuoso_1 = """
1. Resumen Ejecutivo
Línea de Producción 1 muestra un ahorro en Materia Prima de -$54,900 MXN (-8.8%) en enero,
continuando una tendencia de ahorro moderado durante todo el año.

2. Diagnóstico por Centro de Costo
- Línea de Producción 1 / Materia Prima: variación de -$54,900 MXN (-8.8%) en enero.
  Clasificación: ahorro saludable.

3. Alertas de Política
Sin alertas de política en el contexto disponible.

4. Recomendación
Revisar los procesos de compras y gestión de inventarios de Materia Prima.

5. Responsable y Siguiente Paso
Responsable: Jefe de Compras - M. García.
Siguiente paso: realizar un análisis detallado.
"""

indicadores_1 = calcular_indicadores("Materia Prima", -54900, -8.8, [])
resultado_1 = validar_reporte(
    texto_defectuoso_1,
    responsable_esperado="No especificado en los datos proporcionados",
    variacion_mxn=-54900,
    indicadores=indicadores_1,
    contexto_recuperado="",
)
print("--- Validación del Hallazgo 1 (responsable inventado) ---")
print(resultado_1)

In [ ]:
# Hallazgo 2 (Paso 5): confusión entre gasto real y variación al comparar contra el umbral
texto_defectuoso_2 = """
1. Resumen Ejecutivo
La Línea de Producción 2 presenta un sobrecosto en mantenimiento de 24.6% en septiembre.

2. Diagnóstico por Centro de Costo
- Línea de Producción 2 / Mantenimiento: variación de +$17,200 MXN (+24.6%).
  Causa probable: El gasto real ($87,200 MXN) excedió el umbral de $50,000 MXN sobre el
  presupuesto.

3. Alertas de Política
Se activa la regla especial para la categoría de Mantenimiento (POL-FIN-001, sección 3), ya
que el gasto real ($87,200 MXN) excedió el umbral de $50,000 MXN sobre el presupuesto.

4. Recomendación
Solicitar al Gerente de Línea 2 un informe detallado.

5. Responsable y Siguiente Paso
Responsable: Gerente de Línea 2 - M. Torres.
Siguiente paso: presentar el informe.
"""

indicadores_2 = calcular_indicadores("Mantenimiento", 17200, 24.6, [(  "Julio", 18.6), ("Agosto", 20.0)])
contexto_2 = '''(01_Politica_Gasto_Aprobaciones.md)
"Cualquier gasto en la categoría Mantenimiento... que exceda $50,000 MXN sobre el presupuesto
asignado en un mes calendario, requiere aprobación gerencial documentada..." (POL-FIN-001)'''

resultado_2 = validar_reporte(
    texto_defectuoso_2,
    responsable_esperado="Gerente de Línea 2 - M. Torres",
    variacion_mxn=17200,
    indicadores=indicadores_2,
    contexto_recuperado=contexto_2,
)
print("--- Validación del Hallazgo 2 (confusión gasto real vs. variación) ---")
print(resultado_2)

## 5. Los 3 escenarios corregidos, ¿pasan ahora?

Corro los 3 escenarios del Paso 1 con `generar_reporte_validado()` — ya con los problemas de
fondo resueltos en el Paso 5, espero que los tres pasen limpio.


In [ ]:
for nombre, kwargs in [
    ("Escenario 1: Sobrecosto sostenido", dict(
        centro_costo="Línea de Producción 2", categoria="Mantenimiento",
        presupuesto=70000, real=87200,
        historico="Julio +18.6%, Agosto +20.0%, Septiembre +24.6%",
        historico_mensual=[("Julio", 18.6), ("Agosto", 20.0)],
        responsable="Gerente de Línea 2 - M. Torres",
    )),
    ("Escenario 2: Ahorro consistente", dict(
        centro_costo="Línea de Producción 1", categoria="Materia Prima",
        presupuesto=620000, real=565100,
        historico="Variación entre -7% y -9% durante todo el año",
        historico_mensual=[], responsable="Gerente de Línea 1 - R. Hernández",
    )),
    ("Escenario 3: Violación de política puntual", dict(
        centro_costo="Mantenimiento", categoria="Mantenimiento",
        presupuesto=85000, real=144500,
        historico="Pico aislado en julio, no observado en meses anteriores",
        historico_mensual=[], responsable="Coordinador de Mantenimiento - J. Salinas",
    )),
]:
    texto, validacion = generar_reporte_validado(**kwargs)
    print("=" * 90)
    print(nombre, "→", "✅ APROBADO" if validacion["aprobado"] else "⚠️ REVISAR")
    if not validacion["aprobado"]:
        for p in validacion["problemas"]:
            print("  -", p)
    print()
    print(texto)
    print()

---
## Resumen (Paso 7)

Construí un filtro de calidad (`validar_reporte`) que audita, en código, 5 cosas: estructura
de 5 secciones, que el Responsable sea el correcto, que la variación en MXN aparezca, que las
alertas de política sean coherentes con los indicadores, y que no cite políticas
inexistentes. Todo con comparaciones de texto y números, sin volver a llamar al LLM — ya
aprendí en pasos anteriores que un LLM no es buena fuente para revisarse a sí mismo.

Y aquí me pasó algo irónico: mi propio filtro tenía un bug. En una primera corrida daba
falsos positivos — marcaba "excede el umbral" en textos que decían "NO excede" (mi regex no
distinguía negaciones), y tampoco reconocía otras formas del verbo como "excediendo". Lo
arreglé con una función que revisa oración por oración y descarta las negadas.

La Sección 4 confirma que el filtro sí hubiera atrapado mis dos errores anteriores
(responsable inventado, confusión de umbral) si hubiera existido antes. La Sección 5 confirma
que los 3 escenarios ya corregidos pasan limpio, sin falsos positivos.

Siguiente: Paso 8, la interfaz en Streamlit y el despliegue.
